In [0]:
%run ./00_config

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

# Read from Bronze and apply Silver cleansing rules
df_bronze = spark.read.format("delta").load(bronze_path)
df_silver = df_bronze.fillna({"transaction_amount": 0.0}) \
                     .withColumn("cleansed_timestamp", current_timestamp())

# Define the path to the silver container
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/cleansed_sales"

if DeltaTable.isDeltaTable(spark, silver_path):
    # Incremental Upsert (MERGE) into existing Silver Delta table
    silver_table = DeltaTable.forPath(spark, silver_path)
    (
        silver_table.alias("target")
        .merge(
            df_silver.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Success! Incremental MERGE completed at: {silver_path}")
else:
    # First-time load if the Silver Delta table doesn't exist yet
    df_silver.write.format("delta").mode("overwrite").save(silver_path)
    print(f"Success! Initial Silver Delta table created at: {silver_path}")

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(spark, silver_path)

# Compact small files, Z-Order by customer_id, and clean up files older than 7 days
silver_table.optimize().executeZOrderBy("customer_id")
silver_table.vacuum(168)

df_silver_check = spark.read.format("delta").load(silver_path)
print(f"Silver layer merged & optimized! Row count: {df_silver_check.count()}")
display(df_silver_check)